<a id="inicio"></a>
    # AED 03 · KPIs, modelo dimensional y tablero estratégico

    **Núcleo U1/U2 · S05–S06 · RA1 / CE1.4; RA2 / CE2.1–2.2, 2.4** · Colección docente · Edición 2026-10-08.

    Ejecute todas las celdas en orden con el entorno documentado en `../reproducibilidad/README.md`.
    Los controles necesitan kernel activo; las salidas guardadas permiten lectura. Los datos son
    casos docentes locales; las simulaciones y parámetros económicos se identifican como supuestos.
    Consulte la [guía maestra](../guia_maestra_aed.html), el
    [manual científico](../material_propio/AED_manual_cientifico.pdf) y la
    [matriz curricular](../COBERTURA.md). Los ejercicios son formativos.

### Antes de ejecutar

Núcleo · 120 min guiados + 45 min autónomos orientativos.

**Objetivo:** Definir y conciliar una medida con grano, unidad, denominador y responsable.

**Preparación:** Agregaciones, ratios y calidad del módulo 01.

**Ejemplo de referencia:** Ventas de 100 y 900 con márgenes 50 y 90: margen total 140/1000 = 14%, no 30%.

**Práctica:** Filtrar canal y región y conciliar pandas, SQL y la definición DAX.

**Criterio de logro:** Entregar ficha de KPI y tablero con fuente, población, período y acción ante desviaciones.

**Distribución orientativa:** explicación 30 min; práctica guiada 90 min; trabajo autónomo 45 min. Ajustar tras una clase piloto.

[Guía y secuencia](../guia_maestra_aed.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/AED_manual_cientifico.html#sec-kpis)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "_transversal/datos").is_dir())
sys.path.insert(0, str(RAIZ / "04_Analitica_Estrategica_Datos/reproducibilidad"))
from aed_recursos import (
    casapeumo,
    indicadores,
    boldonet,
    plan_produccion,
    decision,
    solar,
    resumen_solar,
    pregunta,
    DATOS,
    SEMILLA,
)

plt.rcParams.update(
    {"figure.figsize": (8, 3.5), "axes.spines.top": False, "axes.spines.right": False}
)

**Práctica importable de BI:** [instrucciones, consulta M, medidas DAX y controles](../material_propio/herramientas_bi/README.md).
La conciliación con Python está disponible; el alumno documenta importación, filtros y resultados del motor que utiliza.

<a id="kpis"></a>
## 1. Un KPI es una definición operativa

Cada indicador necesita fórmula, unidad, población, período, fuente, propietario, meta y acción.
CasaPeumo: margen bruto relativo = suma de margen / suma de ingreso; reclamos por mil = ventas
con reclamo / ventas × 1000. «Bruto» importa: no se incluyen gastos fijos, impuestos ni financiamiento.
Promediar porcentajes por transacción pondera ventas pequeñas igual que grandes y responde otra pregunta.

El tablero debe equilibrar rentabilidad, cliente y operación. Una métrica de vanidad puede mejorar
sin mejorar la decisión: más visitas no implica más contribución. Las metas mensuales del archivo
pertenecen al total del negocio; no se asignan automáticamente a cada canal.

In [2]:
ventas, tablas, auditoria = casapeumo()
display(pd.Series(indicadores(ventas), name="total"))
razon = ventas.Margen.sum() / ventas.Ingreso.sum()
promedio = (ventas.Margen / ventas.Ingreso).mean()
print("Razón de totales:", razon, "Promedio no ponderado:", promedio)
assert np.isclose(razon, np.average(ventas.Margen / ventas.Ingreso, weights=ventas.Ingreso))

ventas                  675.000000
ingreso              101505.611100
margen                30341.401100
margen_pct                0.298914
ticket                  150.378683
reclamos_por_1000        57.777778
entrega_media             0.920000
Name: total, dtype: float64

Razón de totales: 0.29891353562817974 Promedio no ponderado: 0.285147714027597


<a id="modelo-estrella"></a>
## 2. Modelo estrella, grano y medidas

Ventas contiene un registro por venta en este caso; Clientes y Productos tienen claves únicas.
Calendario conecta por fecha. Reclamos es otro hecho, con fecha propia; Marketing y Metas tienen
grano mensual. Unir ventas con filas de marketing por mes multiplica importes: use dimensiones
compartidas y medidas separadas, o agregue antes de unir. Las relaciones van de dimensión a hecho
con filtro simple salvo necesidad justificada. Si se usa `Mes`, cree una dimensión mensual única.

En Power Query: tipifique, documente duplicados exactos, normalice canales y mantenga errores en
una consulta de revisión. En DAX, cree medidas; una columna calculada se evalúa por fila y una
medida según contexto de filtro. No convierta ausencia de datos en cero sin decidir su significado.

In [3]:
import sqlite3

con = sqlite3.connect(":memory:")
ventas[["Canal", "Ingreso", "Margen"]].to_sql("ventas", con, index=False)
sql = pd.read_sql_query(
    "SELECT Canal, SUM(Ingreso) ingreso, SUM(Margen)/NULLIF(SUM(Ingreso),0) margen_pct FROM ventas GROUP BY Canal",
    con,
)
con.close()
py = ventas.groupby("Canal").agg(ingreso=("Ingreso", "sum"), margen=("Margen", "sum"))
assert np.allclose(sql.set_index("Canal").sort_index().ingreso, py.sort_index().ingreso)
display(sql)

,Canal,ingreso,margen_pct
0,Marketplace,23555.9741,0.307762
1,Sitio web,40978.4763,0.298745
2,Tienda física,36971.1607,0.293463


Medidas DAX equivalentes, después de crear `Ingreso` y `Margen` con la misma fórmula:

```text
Ingresos = SUM(Ventas[Ingreso])
Margen bruto = SUM(Ventas[Margen])
Margen % = DIVIDE([Margen bruto], [Ingresos])
Ventas únicas = DISTINCTCOUNT(Ventas[IdVenta])
Ticket = DIVIDE([Ingresos], [Ventas únicas])
```

En Excel use una tabla con referencias estructuradas, una tabla dinámica para sumar y una razón
de sumas. El notebook verifica equivalencia de agregados Python/SQL; DAX y Excel se proporcionan
como receta de implementación, no como archivos PBIX/XLSX ejecutados automáticamente.

In [4]:
@widgets.interact(
    canal=["Todos", *sorted(ventas.Canal.unique())],
    region=["Todas", *sorted(ventas["Región"].unique())],
)
def tablero(canal="Todos", region="Todas"):
    f = ventas.copy()
    if canal != "Todos":
        f = f.loc[f.Canal.eq(canal)]
    if region != "Todas":
        f = f.loc[f["Región"].eq(region)]
    display(pd.Series(indicadores(f), name="KPI"))
    m = f.groupby("Mes").agg(ingreso=("Ingreso", "sum"), margen=("Margen", "sum"))
    fig, ax = plt.subplots()
    m.plot.bar(ax=ax, color=["#176b78", "#bd8841"])
    ax.set(ylabel="UM del caso", title="Resultado por mes; escala desde cero")
    plt.xticks(rotation=0)
    plt.show()
    plt.close(fig)

interactive(children=(Dropdown(description='canal', options=('Todos', 'Marketplace', 'Sitio web', 'Tienda físi…

<a id="retencion"></a>
## 3. Retención y seguimiento de calidad

La retención a 90 días de BoldoNet excluye cohortes inmaduras. Reporte al lado cobertura del
estado, tamaño y rango por faltantes. El costo por participante usa solo costos válidos; excluye
negativos y cuenta esa exclusión. Comparar tasa de grupos no demuestra que el programa causó
la diferencia: la asignación no fue aleatoria. Un dashboard debe hacer visible esa limitación.

In [5]:
c, a = boldonet()
filas = []
for programa, g in c.groupby("Programa_acompañamiento"):
    e = g.loc[g.elegible_kpi]
    m = g.loc[g.maduro]
    activos = e["Estado_90_días"].eq("Activo").sum()
    desconocidos = int((~m.estado_valido).sum())
    costo = g.loc[g.costo_valido, "Costo_acompañamiento"]
    filas.append(
        {
            "programa": programa,
            "clientes": len(g),
            "maduros": len(m),
            "estado_observado": len(e),
            "maduros_sin_estado": desconocidos,
            "cobertura_estado": len(e) / len(m) if len(m) else np.nan,
            "retencion_90d": activos / len(e) if len(e) else np.nan,
            "limite_inferior_faltantes": activos / len(m) if len(m) else np.nan,
            "limite_superior_faltantes": (activos + desconocidos) / len(m) if len(m) else np.nan,
            "costo_valido_medio": costo.mean(),
            "costos_excluidos": int((~g.costo_valido).sum()),
            "actualizacion_antes_ingreso": int(g.actualizacion_antes_ingreso.sum()),
        }
    )
display(pd.DataFrame(filas))

,programa,clientes,maduros,estado_observado,maduros_sin_estado,cobertura_estado,retencion_90d,limite_inferior_faltantes,limite_superior_faltantes,costo_valido_medio,costos_excluidos,actualizacion_antes_ingreso
0,No,135,64,60,4,0.937500,0.716667,0.671875,0.734375,1537.313433,1,10
1,Sí,105,63,61,2,0.968254,0.803279,0.777778,0.809524,29750.000000,1,11


<a id="ejercicio"></a>
## 4. Ejercicio y criterio de revisión

El director pide «un solo número verde». Proponga un tablero de cuatro indicadores, con sus
denominadores y una acción si no cumple. **Solución:** margen bruto %, reclamos por mil, entrega
media y completitud del dato crítico; cada tarjeta tiene período, base y responsable. Use texto
además de color y compare meses homogéneos. Ingresos altos con margen decreciente no justifican
escalar ventas sin investigar descuentos, mezcla y costos.

Contraprueba: dos grupos con ingresos 100 y 900, márgenes 50 y 90 tienen tasas 50% y 10%.
El promedio simple es 30%, pero el negocio tiene 140/1000 = 14%. La diferencia no es redondeo.

In [6]:
ing = np.array([100.0, 900.0])
mar = np.array([50.0, 90.0])
assert np.isclose(mar.sum() / ing.sum(), 0.14)
assert not np.isclose(np.mean(mar / ing), 0.14)
print("Control independiente: margen agregado = 14%")

Control independiente: margen agregado = 14%


<a id="autoevaluacion"></a>
## Autoevaluación
Seleccione y compruebe su respuesta.

In [7]:
auto_03 = pregunta(
    "¿Cómo calcular el margen porcentual agregado?",
    [
        "Promediar los porcentajes de cada venta",
        "Dividir margen total por ingreso total",
        "Sumar los porcentajes",
    ],
    1,
    "La razón de totales equivale a ponderar por ingresos, siempre que las definiciones y el período sean consistentes.",
)

## Práctica de transferencia 03

Dos unidades generan ingresos 100 y 900 UM, con margen 10 y 50 UM. Calcula margen agregado. Diseña cuatro tarjetas con fórmula, período, denominador, responsable y acción; concilia una con la consulta Power Query entregada.

**Entrega:** hipótesis previa, cálculo con unidades, interpretación y límite. Completa tu respuesta antes de consultar la [pauta docente](../material_propio/PAUTA_DOCENTE.md#nb03).

**Discusión:** ¿Cómo comprobarías que la medida cambia correctamente con un filtro?

### Tu resolución

Edita esta celda: escribe tu hipótesis, procedimiento, resultado, interpretación y una comprobación. Adjunta tu código o gráfico si la actividad lo requiere.

**Auto-revisión:** ¿usé la población correcta?, ¿declaré unidades y supuestos?, ¿mi evidencia permite esa conclusión?